# Godot 生态仓库 · 完整数据表 + 排行榜

数据来自 `godot_repos.csv`(由 `export_csv.py` 从 `godot.db` 导出,utf-8-sig)。覆盖 5 个 topic、`star ≥ 1000`、最近 3 个月有推送,去重后 51 个仓库。

**色阶规则**:评分列(`stars` / `forks` / `月均涨星`)按列归一到 0–10 后,红(低)→黄→绿(高);`open_issues` 视为中性计数,右对齐不着色;`full_name` 加粗。
着色用单元格 inline 样式,**GitHub 在线即可见颜色**。

In [1]:
import pandas as pd
from IPython.display import HTML, display

df = pd.read_csv('godot_repos.csv', encoding='utf-8-sig')
TOTAL = len(df)

# 评分列(higher 越好,着红→黄→绿);价格类数字列(右对齐不着色);名称/链接列
SCORE_COLS = ['stars', 'forks', 'stars_per_month']
NUM_PLAIN  = ['open_issues']        # 中性计数,右对齐不着色
BOUNDS = {c: (df[c].min(), df[c].max()) for c in SCORE_COLS}

def score_bg(score):
    """score 为 0-10,返回红→黄→绿的 rgba 背景(套用规范公式)。"""
    ratio = max(0.0, min(1.0, (score - 3) / 7))
    if ratio < 0.5:
        r, g, b = 220, int(60 + ratio * 2 * 180), 60          # 红→黄
    else:
        r = int(220 - (ratio - 0.5) * 2 * 180)
        g = 200
        b = int(60 + (ratio - 0.5) * 2 * 40)                  # 黄→绿
    return f'background-color: rgba({r},{g},{b}, 0.35)'

def to_score10(col, value):
    """把原始指标按列 min-max 归一到 0-10。"""
    lo, hi = BOUNDS[col]
    return 0.0 if hi == lo else (value - lo) / (hi - lo) * 10
print(f'{TOTAL} 行已载入')

51 行已载入


In [2]:
PAGE = 50    # 每页行数(规范 50-80)
DISPLAY_COLS = ['full_name', 'url', 'language', 'stars', 'forks',
                'open_issues', 'stars_per_month']

def render_full_page(frame, start, end, total):
    th = ('background-color:#222;color:#fff;padding:3px 6px;'
          'white-space:nowrap;text-align:left')
    td = 'padding:2px 6px;white-space:nowrap;border-bottom:1px solid #eee'
    out = [f'<h4>第 {start}-{end} / {total}</h4>',
           '<table style="border-collapse:collapse;font-size:9px;'
           'font-family:system-ui,Arial,sans-serif">']
    out.append('<tr><th style="' + th + '">#</th>' +
               ''.join(f'<th style="{th}">{c}</th>' for c in DISPLAY_COLS) + '</tr>')
    for rank, (_, row) in enumerate(frame.iterrows(), start=start):
        cells = [f'<td style="{td};color:#888">{rank}</td>']
        for c in DISPLAY_COLS:
            v = row[c]
            if c == 'full_name':
                cells.append(f'<td style="{td}"><b>{v}</b></td>')
            elif c == 'url':
                cells.append(f'<td style="{td}"><a href="{v}">link</a></td>')
            elif c in SCORE_COLS:
                bg = score_bg(to_score10(c, v))
                txt = f'{v:,.1f}' if c == 'stars_per_month' else f'{int(v):,}'
                cells.append(f'<td style="{td};{bg};text-align:right">{txt}</td>')
            elif c in NUM_PLAIN:
                cells.append(f'<td style="{td};text-align:right">{int(v):,}</td>')
            else:
                cells.append(f'<td style="{td}">{v}</td>')
        out.append('<tr>' + ''.join(cells) + '</tr>')
    out.append('</table>')
    return '\n'.join(out)

html_pages = []
for s in range(0, TOTAL, PAGE):
    page = df.iloc[s:s + PAGE]
    html_pages.append(render_full_page(page, s + 1, min(s + PAGE, TOTAL), TOTAL))
display(HTML('<hr>'.join(html_pages)))

#,full_name,url,language,stars,forks,open_issues,stars_per_month
1,godotengine/godot,link,C++,"112,899","25,726","18,480",755.1
2,Donchitos/Claude-Code-Game-Studios,link,Shell,"22,010","3,181",44,"5,153.7"
3,heroiclabs/nakama,link,Go,"12,778","1,430",124,112.9
4,godotengine/awesome-godot,link,-,"10,210",549,53,76.0
5,Orama-Interactive/Pixelorama,link,GDScript,"9,771",516,83,119.0
6,0xFA11/MultiplayerNetworkingResources,link,C,"8,556",534,2,84.8
7,Redot-Engine/redot-engine,link,C++,"5,880",306,61,56.4
8,dialogic-godot/dialogic,link,GDScript,"5,726",335,168,79.0
9,RodZill4/material-maker,link,GDScript,"5,554",352,311,58.5
10,godotengine/godot-docs,link,reStructuredText,"5,430","3,686","1,077",43.0


In [3]:
def render_ranking(frame, title):
    th = ('background-color:#222;color:#fff;padding:6px 10px;'
          'white-space:nowrap;text-align:left')
    td = 'padding:4px 10px;white-space:nowrap;border-bottom:1px solid #eee'
    cols = ['full_name', 'language', 'stars', 'stars_per_month', 'forks']
    out = [f'<h3>{title}</h3>',
           '<table style="border-collapse:collapse;font-size:11px;'
           'font-family:system-ui,Arial,sans-serif">']
    out.append('<tr><th style="' + th + '">#</th>' +
               ''.join(f'<th style="{th}">{c}</th>' for c in cols) + '</tr>')
    for rank, (_, row) in enumerate(frame.iterrows(), start=1):
        cells = [f'<td style="{td};color:#888">{rank}</td>']
        for c in cols:
            v = row[c]
            if c == 'full_name':
                cells.append(f'<td style="{td}"><b>{v}</b></td>')
            elif c in SCORE_COLS:
                bg = score_bg(to_score10(c, v))
                txt = f'{v:,.1f}' if c == 'stars_per_month' else f'{int(v):,}'
                cells.append(f'<td style="{td};{bg};text-align:right">{txt}</td>')
            else:
                cells.append(f'<td style="{td}">{v}</td>')
        out.append('<tr>' + ''.join(cells) + '</tr>')
    out.append('</table>')
    return HTML('\n'.join(out))

top_stars = df.sort_values('stars', ascending=False).head(15)
render_ranking(top_stars, '★ Stars 排行 Top 15')

#,full_name,language,stars,stars_per_month,forks
1,godotengine/godot,C++,"112,899",755.1,"25,726"
2,Donchitos/Claude-Code-Game-Studios,Shell,"22,010","5,153.7","3,181"
3,heroiclabs/nakama,Go,"12,778",112.9,"1,430"
4,godotengine/awesome-godot,-,"10,210",76.0,549
5,Orama-Interactive/Pixelorama,GDScript,"9,771",119.0,516
6,0xFA11/MultiplayerNetworkingResources,C,"8,556",84.8,534
7,Redot-Engine/redot-engine,C++,"5,880",56.4,306
8,dialogic-godot/dialogic,GDScript,"5,726",79.0,335
9,RodZill4/material-maker,GDScript,"5,554",58.5,352
10,godotengine/godot-docs,reStructuredText,"5,430",43.0,"3,686"


In [4]:
top_growth = df.sort_values('stars_per_month', ascending=False).head(15)
render_ranking(top_growth, '📈 月均涨星 排行 Top 15')

#,full_name,language,stars,stars_per_month,forks
1,Donchitos/Claude-Code-Game-Studios,Shell,"22,010","5,153.7","3,181"
2,htdt/godogen,Python,"3,583",807.9,316
3,godotengine/godot,C++,"112,899",755.1,"25,726"
4,MattParkerDev/SharpIDE,C#,"3,774",510.6,134
5,KsanaDock/Microverse,GDScript,"2,375",283.5,398
6,Coding-Solo/godot-mcp,JavaScript,"4,319",273.9,406
7,Orama-Interactive/Pixelorama,GDScript,"9,771",119.0,516
8,heroiclabs/nakama,Go,"12,778",112.9,"1,430"
9,godot-rust/gdext,Rust,"4,904",103.2,298
10,TokisanGames/Terrain3D,C++,"3,988",97.4,272
